# **Relational Analysis of the IMDb Database**
*Grace Ngo — Data 200: Data Systems for Data Analytics (Fall 2025)*

---

#### **Introduction**

This notebook examines three complementary questions about cinema using an SQLite copy of IMDb: (1) how do audience ratings vary with runtime across genres; (2) how has genre output shifted by decade; and (3) which films define a given year once we account for both rating and audience reach. Each section is driven by a parameterized Python function that performs the core work in SQL—joining `titles` to `ratings`, computing grouping keys, and applying filters—before rendering compact, readable figures in matplotlib/Seaborn. The result is a sequence of visual explanations that a general reader can follow without prior knowledge of SQL or Python, while still being faithful to the relational model.

#### **Runtime and Audience Reception Across Genres**

The first function, `runtime_vs_rating_by_genre(connection, genres, min_votes, min_runtime, max_runtime)`, asks a direct question: Do longer films earn higher ratings and does the answer depend on genre? The query at its heart joins `titles` and `ratings` via `USING(title_id)`, restricts the sample to non-adult movies within a user-specified runtime band (e.g., 40–200 minutes), and applies a vote threshold so that the analysis reflects widely seen titles. Crucially, the function bins runtime in SQL, not in pandas, by computing a ten-minute grouping key: `ROUND(t.runtime_minutes / 10) * 10 AS runtime_bin`. Grouping and averaging are also pushed to SQL, so each result row represents one runtime bin for one genre, with the corresponding average rating.

The figure produced an eight-panel grid covering Comedy, Drama, Action, Romance, Thriller, Documentary, Horror, and Adventure using a consistent axis scale for fair comparison. In every panel, pale points show the underlying titles while a red polyline connects the bin-level averages. Read from left to right, the picture is nuanced and genre-specific. Comedy rises modestly from the 90-minute neighborhood toward the 110–130 minute range, hinting that slightly longer comedies may benefit from additional narrative room without feeling overlong. Drama sustains high ratings across a wide span and often crests between 140 and 160 minutes, consistent with the conventions of prestige storytelling. Horror performs best compactly, roughly 90–110 minutes, after which trendlines flatten or slip, a pattern aligned with pacing expectations in suspense-driven films. Action and Adventure show greater dispersion: some bins are excellent, others middling, reflecting the variety of subgenres (ensemble epics, lean thrillers, effects-heavy franchises).

Two choices make this function useful beyond the classroom. First, the vote threshold `(r.votes >= :min_votes)` reduces noise from little-seen titles. Second, the binning in SQL ensures that the “heavy lifting”—grouping and aggregation—happens close to the data. For a programming or acquisitions team, the output provides runtime guidance by genre: if a cut is ballooning past 150 minutes in horror, or shrinking below 100 minutes in drama, the trendlines provide an empirical frame for editorial decisions.

#### **Genre Popularity by Decade**

The second function, `genre_popularity_heatmap(connection, genres, decade_start, decade_end)`, turns from reception to supply. It asks: What did the industry make, and when? The SQL extracts every movie between a start and end year and computes the release decade directly in the query: `(t.premiered / 10) * 10 AS decade`. Because IMDb stores multiple genres in a comma-delimited string, the function expands each record so a film with “Comedy,Romance” counts toward both Comedy and Romance. The result is reshaped into a `genre × decade` matrix of counts and rendered as a heatmap with annotated values.

Viewed across a century, the pattern is striking. Drama is prolific in every era, with counts climbing dramatically in the 2000s and peaking in the 2010s. Documentary evolves from a modest presence early in the record to a major contributor in recent decades, reflecting the rise of nonfiction storytelling and the economics of streaming. Action and Adventure expand sharply from the 1980s onward, mirroring the blockbuster era and the economics of franchises. Horror, scarcely present in the 1920s, becomes common from the 1970s forward. Because this figure shows counts, not ratings, it complements the first chart: the heatmap answers how much was made, while the runtime chart hints at how those films were received.

Two implementation details matter for interpretation. First, by retaining titles with multiple genres and counting them in each applicable row, the visualization reflects how modern films straddle categories (e.g., Action-Adventure, Comedy-Drama). Second, the decade computation is done in SQL, ensuring reproducible, schema-level logic rather than ad-hoc parsing downstream. For curators, educators, or catalog teams, the heatmap serves as a dashboard for coverage: which decades are rich for a given genre, and where might a library or syllabus be thin?

#### **Year-Specific Standouts: Top 25 Movies**

The final function, `top_movies_of_year(connection, year, min_votes, top_n)`, focuses on a single year and blends quality (rating) with consensus (votes). A Common Table Expression gathers candidate films via a join of  `titles ` and  `ratings `, filters to the chosen year and to non-adult content, and enforces a vote floor (e.g., 5,000) so the list reflects widely seen work. The outer query orders by  `rating DESC`, votes DESC and returns the top N. Back in Python, the first token of the genre string is treated as a “main genre” for color encoding, and a horizontal bar chart displays each title’s rating with an inline annotation of its vote count. The legend maps colors to genres.

For 2010, the chart tells a balanced story. Inception sits near the top with an 8.8 rating and a vast vote count, an archetypal crowd-and-critic success. Yet nearby are Senna and Inside Job, documentaries with excellent scores and far smaller audiences, reminding us that high quality is not the sole province of blockbusters. Animation (Toy Story 3), psychological thrillers (Shutter Island), and international dramas (Incendies) sit side by side. The genre coloring makes that diversity legible at a glance, while the vote counts keep a sense of scale: some titles are widely loved; others are intensely admired by niche audiences.

From a product or editorial standpoint, this function is immediately actionable. Its parameterization allows a weekly or monthly “year in review” feature; the vote threshold offers control over volatility; and the SQL ordering guarantees deterministic reproduction. Because the function returns the underlying table alongside the plot, the same list can populate recommendation modules, press notes, or program slates.

#### **Method**

Across all three functions, parameters are exposed so the same logic generalizes without code edits: genres can be swapped, year ranges adjusted, vote floors tuned. Null handling ensures that the required fields (runtime, rating, premiered year) are present before inclusion. Most importantly, the relational work lives in SQL: joins, computed columns (e.g., runtime bins, decades), grouping, and ordering occur in the database. Python’s role is to issue parameterized queries securely and to present the results in figures with consistent scales, axis labels, and legends. This division of labor keeps the notebook efficient, auditable, and easy to reason about.

Visually, the choices are deliberate. Constant axes in the runtime grid allow true cross-genre comparison. The decade heatmap uses both color intensity and numeric annotations so a casual reader can read counts without guessing. The top-movies bar chart is horizontal—long titles remain legible—and annotations place the most relevant numbers directly on the bars, avoiding cross-referencing. Pastel palettes and restrained styling keep the emphasis on the data.

#### **Conclusion**

Taken together, the three visualizations form a coherent narrative about films: how long high-performing titles tend to run within each genre; how the industry’s genre mix has shifted by decade; and which films, in a given year, distinguish themselves once we require both strong ratings and meaningful audience size. The functions meet the project’s requirements—multi-table SQL joins, grouping and aggregation in SQL, parameterized design, and clear visualizations—while remaining readable to a general audience. More importantly, they are practically useful: the runtime curves inform editing and programming; the heatmap informs catalog strategy; and the yearly top-25 list is presentation-ready for discovery features or retrospectives. As a whole, the notebook demonstrates disciplined use of the relational model, careful parameterization, and visual storytelling that translates raw data into explanations a broad audience can trust.